# TEST SINGLE AUGMENTATION

In [ ]:
from pathlib import Path
import numpy as np
import torch
import io
from PIL import Image
import cv2

from torch.utils.data import DataLoader
from emma import EmmaDataset, EmmaSearcher, SearchQuery
from emma.dataloader import EmmaMapDataset
from emma.dataloader import emma_collate

import matplotlib.pyplot as plt

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"], require_listed=["mass"])
asins = se.query_asins(query)
print(len(asins))

In [ ]:
# SELECT A SAMPLE
rec = ds.get_record(asins[0])
image_dict = ds.get_images(rec)
first_img_key, first_img_bytes = next(iter(image_dict.items()))
first_img = Image.open(io.BytesIO(first_img_bytes)).convert("RGB")
first_img_np = np.array(first_img)
first_img.show()

## > AUGMENTATION

In [ ]:
def thresholding_segmentation(image_rgb, threshold, kernel_size, open_iterations):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    _, bin_img = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY_INV) # cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    bin_img = cv2.morphologyEx(bin_img, cv2.MORPH_OPEN, kernel, iterations=open_iterations)

    valid_mask = (bin_img > 127).astype(np.uint8, copy=False)

    return valid_mask

In [ ]:
segmentation = thresholding_segmentation(first_img_np, 253, 3, 2)

## > VISUALIZE

In [ ]:
plt.imshow(segmentation, cmap='gray', interpolation='nearest')
plt.axis('off')
plt.show()